In [ ]:
import pandas as pd

# Pull the sheet directly — no download needed
url = "https://docs.google.com/spreadsheets/d/1n9dlO0Y1WdON5cyc8jf82M6Jn8JLP2dha_FPzcx5AuQ/export?format=csv&gid=0"
df = pd.read_csv(url)
print(df.head())


                               Data Source  \
0       Feeding America – Map the Meal Gap   
1                Maryland open data portal   
2                      DC open data portal   
3                Virginia open data portal   
4  Prince George’s County open data portal   

                                      Link  \
0           https://map.feedingamerica.org   
1            https://opendata.maryland.gov   
2                 https://opendata.dc.gov/   
3               https://data.virginia.gov/   
4  https://data.princegeorgescountymd.gov/   

                                         Description  \
0  Provides county level food insecurity rates, n...   
1  Statewide datasets including demographics, sch...   
2  Contains datasets on community resources, food...   
3  Includes datasets related to community service...   
4  Includes local datasets such as schools, publi...   

                How NourishNet uses this data source Added By Unnamed: 5  
0  NourishNet uses this to identif

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

sheet_id = "1n9dlO0Y1WdON5cyc8jf82M6Jn8JLP2dha_FPzcx5AuQ"
response = requests.get(f"https://docs.google.com/spreadsheets/d/{sheet_id}/edit")
gids = re.findall(r'gid=(\d+)', response.text)
gids = list(set(gids))  # remove duplicates
print("Found gids:", gids)

Found gids: []


In [ ]:
import pandas as pd

all_dfs = []

# Check if gids is empty and default to ['0'] if it is,
# assuming there's at least a default sheet at gid=0.
# This addresses the issue of gids being an empty list
# preventing the loop from running.
if not gids:
    print("Warning: 'gids' list is empty. Defaulting to gid='0'.")
    gids = ['0']

for gid in gids:
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    try:
        temp_df = pd.read_csv(url)
        temp_df['source_tab'] = gid
        all_dfs.append(temp_df)
        print(f"✅ gid {gid}: {len(temp_df)} rows")
    except Exception as e:
        print(f"❌ gid {gid} failed: {e}")

if all_dfs: # Only concatenate if there's data to concatenate
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal rows: {len(df_all)}")
    print(df_all.head(10))
else:
    df_all = pd.DataFrame() # Create an empty DataFrame if no data was loaded
    print("\nNo data loaded from any GID. 'df_all' is an empty DataFrame.")

✅ gid 0: 41 rows

Total rows: 41
                                    Data Source  \
0            Feeding America – Map the Meal Gap   
1                     Maryland open data portal   
2                           DC open data portal   
3                     Virginia open data portal   
4       Prince George’s County open data portal   
5               Capital Area Food Bank Heat Map   
6                 Maryland Food Bank Hunger Map   
7                  Maryland Food Insecurity Map   
8                   USDA Food Environment Atlas   
9  US Census Bureau – American Community Survey   

                                                Link  \
0                     https://map.feedingamerica.org   
1                      https://opendata.maryland.gov   
2                           https://opendata.dc.gov/   
3                         https://data.virginia.gov/   
4            https://data.princegeorgescountymd.gov/   
5               https://www.capitalareafoodbank.org/   
6          ht

In [ ]:
# Find whichever column has the URLs
for col in df_all.columns:
    if df_all[col].astype(str).str.contains('http').any():
        print(f"Column '{col}' has links:")
        links = df_all[col].dropna().tolist()
        for i, l in enumerate(links):
            print(f"  {i}: {l}")

Column 'Link' has links:
  0: https://map.feedingamerica.org
  1: https://opendata.maryland.gov
  2: https://opendata.dc.gov/
  3: https://data.virginia.gov/
  4: https://data.princegeorgescountymd.gov/
  5: https://www.capitalareafoodbank.org/
  6: https://mdfoodbank.org/hunger-in-maryland
  7: https://mda.maryland.gov/
  8: https://www.ers.usda.gov/data-products/food-environment-atlas
  9: https://data.census.gov/
  10: https://extension.umd.edu/resource/food-access-resources/
  11: https://www.epa.gov/sustainable-management-food/excess-food-opportunities-map
  12: https://compass.maryland.gov/map/
  13: https://princegeorges.maps.arcgis.com/apps/dashboards/9f9202c51cc345ab9e0e1aa21a23bb76
  14: https://pgcfec.org/resources/find-food-food-pantry-listings/
  15: https://msa.maryland.gov/megafile/msa/speccol/sc5300/sc5339/000113/014000/014769/unrestricted/20120605e-021.pdf
  16: https://mdfoodbank.org/find-food/#more
  17: https://search.211md.org/search?query=food+pantry&query_label=f

In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# Test with Capital Area Food Bank (link 5)
test_url = "https://www.capitalaareafoodbank.org/"
try:
    r = requests.get(test_url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, 'html.parser')
    print("Status:", r.status_code)
    print("Title:", soup.title.text.strip() if soup.title else "None")
    print("\nText preview:")
    print(soup.get_text()[:500])
except Exception as e:
    print("Error:", e)

Error: HTTPSConnectionPool(host='www.capitalaareafoodbank.org', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7cc677efc1a0>: Failed to resolve 'www.capitalaareafoodbank.org' ([Errno -2] Name or service not known)"))


In [ ]:
import json
import time

links = [
    "https://map.feedingamerica.org",
    "https://opendata.maryland.gov",
    "https://opendata.dc.gov/",
    "https://data.virginia.gov/",
    "https://data.princegeorgescountymd.gov/",
    "https://www.capitalaareafoodbank.org/",
    "https://mdfoodbank.org/hunger-in-maryland",
    "https://mda.maryland.gov/",
    "https://www.capitalaareafoodbank.org/",
    "https://pgcfec.org/resources/find-food-food-pantry-listings/",
    "https://carolinebettertogether.org/food-pantries",
    "https://mocofoodcouncil.org/map/",
    "https://search.211md.org/search?query=food+pantry"
]

results = []

for url in links:
    try:
        r = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')

        text = soup.get_text(separator=' ', strip=True)

        results.append({
            "url": url,
            "title": soup.title.text.strip() if soup.title else "",
            "status": r.status_code,
            "text_preview": text[:1000],  # first 1000 chars
            "success": True
        })
        print(f"✅ {url}")
        time.sleep(1)  # be polite, don't hammer servers

    except Exception as e:
        results.append({"url": url, "success": False, "error": str(e)})
        print(f"❌ {url} — {e}")

print(f"\nDone! {sum(r['success'] for r in results)}/{len(results)} succeeded")

✅ https://map.feedingamerica.org
✅ https://opendata.maryland.gov
✅ https://opendata.dc.gov/
✅ https://data.virginia.gov/
✅ https://data.princegeorgescountymd.gov/
❌ https://www.capitalaareafoodbank.org/ — HTTPSConnectionPool(host='www.capitalaareafoodbank.org', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7cc677d839b0>: Failed to resolve 'www.capitalaareafoodbank.org' ([Errno -2] Name or service not known)"))
✅ https://mdfoodbank.org/hunger-in-maryland
✅ https://mda.maryland.gov/
❌ https://www.capitalaareafoodbank.org/ — HTTPSConnectionPool(host='www.capitalaareafoodbank.org', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7cc6776513d0>: Failed to resolve 'www.capitalaareafoodbank.org' ([Errno -2] Name or service not known)"))
✅ https://pgcfec.org/resources/find-food-food-pantry-listings/
✅ https://carolinebettertogether.org/food-pa

In [ ]:
with open('data.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Saved to data.json!")
print(f"Total entries: {len(results)}")

# Download it directly to your computer
from google.colab import files
files.download('data.json')

✅ Saved to data.json!
Total entries: 13


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

pantries = []
page = 1

while True:
    url = f"https://search.211md.org/search?query=food+pantry&page={page}"
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, 'html.parser')

    # Find all result cards
    cards = soup.find_all('div', class_='result')
    if not cards:
        print(f"No more results at page {page}, stopping.")
        break

    for card in cards:
        name = card.find('h2') or card.find('h3')
        address = card.find(class_='address') or card.find('address')
        phone = card.find(class_='phone')
        desc = card.find(class_='description') or card.find('p')

        pantries.append({
            "name": name.text.strip() if name else "",
            "address": address.text.strip() if address else "",
            "phone": phone.text.strip() if phone else "",
            "description": desc.text.strip() if desc else "",
            "source": "211md.org"
        })

    print(f"✅ Page {page}: {len(cards)} results")
    page += 1
    time.sleep(1)

print(f"\nTotal pantries scraped: {len(pantries)}")

No more results at page 1, stopping.

Total pantries scraped: 0


In [ ]:
with open('food_data.json', 'w') as f:
    json.dump(pantries, f, indent=2)

print(f"✅ Saved {len(pantries)} pantries to food_data.json")

from google.colab import files
files.download('food_data.json')

✅ Saved 0 pantries to food_data.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

url = "https://search.211md.org/search?query=food+pantry"
r = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(r.text, 'html.parser')

# Print raw HTML to see structure
print(r.status_code)
print(soup.prettify()[:3000])  # first 3000 chars of HTML

200
<!DOCTYPE html>
<html class="max-sm:!text-[100%]" lang="en" style="--primary:206.48275862068965 100% 28.431372549019606%;--primary-foreground:0 0% 100%;--secondary:33.79310344827587 100% 65.88235294117646%;--secondary-foreground:0 0% 0%;--border-radius:6px;--header-start:#ffffff;--header-end:#ffffff">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1, maximum-scale=1" name="viewport"/>
  <link data-precedence="next" href="/_next/static/css/d9f2998739b9294e.css" rel="stylesheet"/>
  <link data-precedence="next" href="/_next/static/css/f4be61d25765a56f.css" rel="stylesheet"/>
  <link as="script" fetchpriority="low" href="/_next/static/chunks/webpack-e5536b107badbcae.js" rel="preload"/>
  <script async="" src="/_next/static/chunks/4bd1b696-cc729d47eba2cee4.js">
  </script>
  <script async="" src="/_next/static/chunks/5964-8536500dcc67574f.js">
  </script>
  <script async="" src="/_next/static/chunks/main-app-e80e8e96c2b08c40.js">
  </script>
  <scr

In [ ]:
import requests

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# Try common API patterns for 211 sites
api_urls = [
    "https://search.211md.org/api/search?query=food+pantry",
    "https://search.211md.org/api/resources?query=food+pantry",
    "https://search.211md.org/_next/data/search?query=food+pantry",
    "https://api.211md.org/search?query=food+pantry",
]

for url in api_urls:
    try:
        r = requests.get(url, headers=headers, timeout=10)
        print(f"Status {r.status_code}: {url}")
        if r.status_code == 200:
            print(r.text[:500])
            print("---")
    except Exception as e:
        print(f"Failed: {url} — {e}")

Status 404: https://search.211md.org/api/search?query=food+pantry
Status 404: https://search.211md.org/api/resources?query=food+pantry
Status 404: https://search.211md.org/_next/data/search?query=food+pantry
Failed: https://api.211md.org/search?query=food+pantry — HTTPSConnectionPool(host='api.211md.org', port=443): Max retries exceeded with url: /search?query=food+pantry (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7cc67637b3e0>: Failed to resolve 'api.211md.org' ([Errno -5] No address associated with hostname)"))


In [ ]:
import requests
import json

headers = {'User-Agent': 'Mozilla/5.0'}

# HelpSteps is the open API behind many 211 sites
url = "https://www.helpsteps.com/api/v1/resources?keyword=food+pantry&location=Maryland&radius=50"

r = requests.get(url, headers=headers, timeout=10)
print("Status:", r.status_code)
print(r.text[:1000])

Status: 200
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <link rel="icon" type="image/png" href="/favicon.ico" />
    <link rel="apple-touch-icon" href="/helpsteps-icon.webp" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <meta name="theme-color" content="#00294f" />
    <meta name="description" content="Find local health and human services resources in Massachusetts with 211 HELPSteps" />
    <link rel="manifest" href="/manifest.json" />
    <link rel="preconnect" href="https://maps.googleapis.com" />
    <link rel="preconnect" href="https://maps.gstatic.com" crossorigin />
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500;700&display=swap" rel="stylesheet">
    <title>211 HELPSteps</title>
    <script type="module" crossorigin src="/assets/index-CTV6B1k

In [ ]:
# FindHelp.org has a public search for MD food resources
url = "https://api.findhelp.com/api/v1/resources?query=food+pantry&location=Maryland"
r = requests.get(url, headers=headers, timeout=10)
print("Status:", r.status_code)
print(r.text[:500])

Status: 404
<!DOCTYPE html>
<html lang="en">
<head>

	<title>
	Page Not Found
	</title>

	

	<meta http-equiv="X-UA-Compatible" content="IE=edge">
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1">

	<link rel="preconnect" href="https://fonts.googleapis.com">
	<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
	<link rel="stylesheet" href="https://fonts.googleapis.com/css?family=Source+Sans+Pro:300,400,400i,600,700:ital&amp;display=swap">
	<link r


In [ ]:
# OpenStreetMap Overpass API - completely free, no key needed
# Searches for food banks in DC/MD area
overpass_url = "https://overpass-api.de/api/interpreter"

query = """
[out:json];
(
  node["amenity"="food_bank"](38.5,-77.5,39.5,-76.5);
  node["social_facility"="food_bank"](38.5,-77.5,39.5,-76.5);
  node["amenity"="soup_kitchen"](38.5,-77.5,39.5,-76.5);
);
out body;
"""

r = requests.post(overpass_url, data=query, timeout=30)
print("Status:", r.status_code)
data = r.json()
print(f"Found {len(data['elements'])} locations")
print(json.dumps(data['elements'][:2], indent=2))

Status: 200
Found 48 locations
[
  {
    "type": "node",
    "id": 367141197,
    "lat": 38.9328594,
    "lon": -77.0368238,
    "tags": {
      "addr:state": "DC",
      "amenity": "social_facility",
      "ele": "61",
      "gnis:feature_id": "2457456",
      "name": "Canaan Baptist Church Coalition for the Homeless",
      "social_facility": "food_bank",
      "source": "USGS Geonames"
    }
  },
  {
    "type": "node",
    "id": 367141203,
    "lat": 38.9103,
    "lon": -77.0217,
    "tags": {
      "addr:city": "Washington",
      "addr:housenumber": "1525",
      "addr:postcode": "20001",
      "addr:state": "DC",
      "addr:street": "7th Street Northwest",
      "amenity": "social_facility",
      "ele": "25",
      "name": "Bread for the City",
      "opening_hours": "Mo-Th 09:00-17:00; Fr 09:00-12:00",
      "phone": "+1-202-265-2400",
      "social_facility": "food_bank",
      "website": "https://breadforthecity.org/"
    }
  }
]


In [ ]:
import json

# Clean the raw OpenStreetMap data into a nice format
cleaned = []

for element in data['elements']:
    tags = element.get('tags', {})

    # Build address
    house = tags.get('addr:housenumber', '')
    street = tags.get('addr:street', '')
    city = tags.get('addr:city', '')
    state = tags.get('addr:state', '')
    postcode = tags.get('addr:postcode', '')
    address = f"{house} {street}, {city}, {state} {postcode}".strip(', ')

    cleaned.append({
        "name": tags.get('name', 'Unknown'),
        "lat": element.get('lat'),
        "lon": element.get('lon'),
        "address": address,
        "phone": tags.get('phone', tags.get('contact:phone', '')),
        "website": tags.get('website', tags.get('contact:website', '')),
        "opening_hours": tags.get('opening_hours', ''),
        "type": tags.get('social_facility', tags.get('amenity', '')),
        "source": "OpenStreetMap"
    })

print(f"Total locations: {len(cleaned)}")
print(json.dumps(cleaned[:3], indent=2))

Total locations: 48
[
  {
    "name": "Canaan Baptist Church Coalition for the Homeless",
    "lat": 38.9328594,
    "lon": -77.0368238,
    "address": "DC",
    "phone": "",
    "website": "",
    "opening_hours": "",
    "type": "food_bank",
    "source": "OpenStreetMap"
  },
  {
    "name": "Bread for the City",
    "lat": 38.9103,
    "lon": -77.0217,
    "address": "1525 7th Street Northwest, Washington, DC 20001",
    "phone": "+1-202-265-2400",
    "website": "https://breadforthecity.org/",
    "opening_hours": "Mo-Th 09:00-17:00; Fr 09:00-12:00",
    "type": "food_bank",
    "source": "OpenStreetMap"
  },
  {
    "name": "Little Free Pantry",
    "lat": 39.2250178,
    "lon": -76.5919934,
    "address": "1321 Filbert St, Baltimore, MD",
    "phone": "",
    "website": "",
    "opening_hours": "",
    "type": "food_bank",
    "source": "OpenStreetMap"
  }
]


In [ ]:
# Get MORE results - expand to all of MD/DC/VA area
query2 = """
[out:json];
(
  node["amenity"="food_bank"](38.0,-77.8,39.8,-76.0);
  node["social_facility"="food_bank"](38.0,-77.8,39.8,-76.0);
  node["amenity"="soup_kitchen"](38.0,-77.8,39.8,-76.0);
  node["social_facility"="food_pantry"](38.0,-77.8,39.8,-76.0);
  node["amenity"="community_centre"]["food"="yes"](38.0,-77.8,39.8,-76.0);
  way["social_facility"="food_bank"](38.0,-77.8,39.8,-76.0);
  way["social_facility"="food_pantry"](38.0,-77.8,39.8,-76.0);
);
out center body;
"""

r2 = requests.post(overpass_url, data=query2, timeout=30)
data2 = r2.json()
print(f"Found {len(data2['elements'])} total locations")

# Clean this bigger dataset
cleaned2 = []
for element in data2['elements']:
    tags = element.get('tags', {})
    # Handle both nodes and ways (ways have 'center')
    lat = element.get('lat') or (element.get('center') or {}).get('lat')
    lon = element.get('lon') or (element.get('center') or {}).get('lon')

    house = tags.get('addr:housenumber', '')
    street = tags.get('addr:street', '')
    city = tags.get('addr:city', '')
    state = tags.get('addr:state', '')
    postcode = tags.get('addr:postcode', '')
    address = f"{house} {street}, {city}, {state} {postcode}".strip(', ')

    cleaned2.append({
        "name": tags.get('name', 'Unknown'),
        "lat": lat,
        "lon": lon,
        "address": address,
        "phone": tags.get('phone', tags.get('contact:phone', '')),
        "website": tags.get('website', tags.get('contact:website', '')),
        "opening_hours": tags.get('opening_hours', ''),
        "type": tags.get('social_facility', tags.get('amenity', '')),
        "source": "OpenStreetMap"
    })

print(f"Cleaned: {len(cleaned2)} locations")

Found 62 total locations
Cleaned: 62 locations


In [ ]:
with open('food_locations.json', 'w') as f:
    json.dump(cleaned2, f, indent=2)

print(f"✅ Saved {len(cleaned2)} food locations!")

from google.colab import files
files.download('food_locations.json')

✅ Saved 62 food locations!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 18 — Re-run anytime for fresh data from OpenStreetMap
# This is the "real-time ingestion" mechanism

import datetime
print(f"Data last refreshed: {datetime.datetime.now()}")
print(f"Total locations: {len(cleaned2)}")
print("To update: Re-run this entire notebook from top to bottom")
print("Output: food_locations.json — ready to use in the React app")

Data last refreshed: 2026-04-12 23:57:29.099970
Total locations: 62
To update: Re-run this entire notebook from top to bottom
Output: food_locations.json — ready to use in the React app
